# Downstream classification sobre TRANSFER_TARGET primary "limpios"

Lanza los 3 modos (`from_scratch`, `linear_probing`, `full_finetuning`
con `lr_backbone=1e-5`) sobre cada TT cuyo target sea classification.

Candidatos iniciales:

- HSG18 (hdd) — target fault, classification
- CALCE_CS2 (batteries) — target rul, **regresion (se filtra)**
- PHM18 (wind) — target fault, classification
- PBCP16 (small) — target fault, classification
- IEEE14 (mosfets_power) — target rul35, **regresion (se filtra)**

Por ahora solo classification (3 datasets esperados). CMAPSS, CALCE_CS2
e IEEE14 (RUL) esperan a tener cabeza RUL y a resolver semantica.

## Politica de commit / push

**Por defecto este notebook NO commitea ni pushea**. El usuario autoriza
manualmente. Cambiar `AUTO_COMMIT` y `AUTO_PUSH` a True en la celda de
setup solo si se quiere automatizar la persistencia tras una corrida.

## Como usar este notebook

1. Runtime > Change runtime type > GPU A100 (recomendado).
2. Run All. ~6-9 h estimado segun tamano de cada dataset.
3. Si la VM se mata: vuelve a abrir el notebook, Run All otra vez.
   El bucle principal **es reentrante**: salta los runs que ya tienen
   `run_info.json` en Drive.
4. Si algun dataset falla individualmente (target ilegible, OOM, etc.)
   se anota como FAIL y el bucle sigue con el siguiente.
5. Al final, celda de resumen + (si AUTO_COMMIT=True) commit + push.

## Hiperparametros (replica el ganador de CWRU)

| modo | lr_head | lr_backbone | notas |
|---|---|---|---|
| from_scratch | 1e-3 | 1e-4 | mismo que la config base de CWRU |
| linear_probing | 1e-3 | (irrelevante, backbone congelado) | |
| full_finetuning | 1e-3 | **1e-5** | sweet spot detectado en CWRU |

`max_epochs=20`, `batch_size=64` adaptativo (`B*C <= 512`),
`metric_for_best=macro_f1_val`. AMP auto en CUDA.


In [ ]:
# Setup 1/2: montar Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Setup 2/2: clonar/actualizar repo, instalar deps, configurar git
!bash /content/drive/MyDrive/fm_fl_phmd/colab_init.sh

# Cada VM Colab arranca sin git user configurado. Lo seteamos local al
# repo para que el commit final (si AUTO_COMMIT=True) no falle por
# 'Author identity unknown'. Estos valores son los que el TFM usa en
# commits previos.
!cd /content/fm_fl_phmd && git config user.email "josesanzduran@gmail.com" && git config user.name "Jose Sanz"


In [ ]:
# Cambiar al repo y traer cambios pendientes
%cd /content/fm_fl_phmd
!git pull
!git log --oneline -3

# --------------------------------------------------------------------
# Flags de control de commit/push
# --------------------------------------------------------------------
# Por defecto, este notebook NO commitea ni pushea. La persistencia de
# resultados queda controlada manualmente por el usuario. Activa estos
# flags solo si quieres que la ultima celda haga commit + push real.
AUTO_COMMIT = False
AUTO_PUSH = False
print(f"AUTO_COMMIT={AUTO_COMMIT}  AUTO_PUSH={AUTO_PUSH}")


In [ ]:
# Pre-check robusto de los datasets candidatos.
#
# El manifest de harmonization NO tiene un campo 'task' (eso es del
# config YAML del trainer). El tipo de tarea lo inferimos:
#   1) manifest['task'] si por casualidad lo trae.
#   2) manifest['tipo_target'] si esta.
#   3) heuristica sobre target_col: rul/soh/soc/capacity/voltage/...
#      -> regresion, lo demas -> classification.
#
# Datasets que pasen el pre-check entran a ELIGIBLE. El resto queda SKIP.
import json
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/fm_fl_phmd")
SSL_CKPT = DRIVE / "checkpoints/ssl_central_full/ssl_central_full_patchtst_phm/ckpt_step100000.pt"
PROCESSED = DRIVE / "processed"

CANDIDATES = ["HSG18", "CALCE_CS2", "PHM18", "PBCP16", "IEEE14"]

assert SSL_CKPT.is_file(), f"NO existe SSL ckpt: {SSL_CKPT}"
print(f"SSL ckpt OK: {SSL_CKPT}  ({round(SSL_CKPT.stat().st_size/1024/1024, 1)} MB)\n")


def es_classification(manifest: dict, target_col: str):
    """Devuelve (es_classification: bool, motivo: str)."""
    # 1) Campo explicito si existiera
    t = manifest.get("task")
    if t:
        return (t == "classification_multiclass", f"task={t}")
    tt = manifest.get("tipo_target")
    if tt:
        tt_low = str(tt).lower()
        if "rul" in tt_low or "regress" in tt_low:
            return (False, f"tipo_target={tt}")
        return (True, f"tipo_target={tt}")
    # 2) Heuristica sobre target_col
    tc = str(target_col).lower()
    regression_keywords = ("rul", "soh", "soc", "capacity", "voltage",
                            "current", "temperature", "remaining", "life")
    if any(k in tc for k in regression_keywords):
        return (False, f"target_col='{target_col}' parece regresion")
    return (True, f"target_col='{target_col}' parece classification")


ELIGIBLE = []
print(f"{'dataset':<12} {'target_col':<14} {'n_units':>8} {'n_canales':>10} {'n_windows':>12}   estado")
print("-" * 110)
for d in CANDIDATES:
    base = PROCESSED / d
    mpath = base / "manifest.json"
    if not mpath.is_file():
        print(f"{d:<12} -              -        -          -            SKIP no_manifest")
        continue
    try:
        m = json.load(open(mpath))
    except Exception as e:
        print(f"{d:<12} -              -        -          -            SKIP manifest_invalido: {e}")
        continue
    target_col = m.get("target_col", "?")
    n_units = m.get("n_units_total", "?")
    n_canales = m.get("n_channels", m.get("n_canales", "?"))
    nw = m.get("n_windows_por_split") or {}
    if isinstance(nw, dict) and nw:
        n_windows = sum(int(v) for v in nw.values())
    else:
        n_windows = m.get("n_windows", "?")

    # Verificar shards reales
    n_shards = {}
    for s in ("train", "val", "test"):
        n_shards[s] = len(list((base / s).glob("*.tar"))) if (base / s).is_dir() else 0
    shards_ok = all(n > 0 for n in n_shards.values())

    is_class, motivo = es_classification(m, target_col)
    if not is_class:
        estado = f"SKIP regresion ({motivo})"
    elif not shards_ok:
        estado = f"SKIP shards={n_shards}"
    else:
        estado = f"OK ({motivo})"
        ELIGIBLE.append(d)

    print(f"{d:<12} {str(target_col):<14} {str(n_units):>8} {str(n_canales):>10} {str(n_windows):>12}   {estado}")

print(f"\nDatasets elegibles para el bucle: {ELIGIBLE}")
print(f"({len(ELIGIBLE)} elegibles x 3 modos = {len(ELIGIBLE)*3} corridas)")


In [ ]:
# Sanity tests rapidos (~5 s)
!python -m pytest tests/test_downstream_metrics.py \
                  tests/test_downstream_pooling.py \
                  tests/test_downstream_classification_head.py \
                  tests/test_downstream_adaptive_batch.py -q


In [ ]:
# Helpers para generar configs y lanzar runs
import os
import shutil
import subprocess
import time
import json
from pathlib import Path

REPO = Path("/content/fm_fl_phmd")
LOG_DIR_ROOT = DRIVE / "logs/downstream"
CKPT_DIR_ROOT = DRIVE / "checkpoints/downstream"

os.chdir(REPO)

BASE_CONFIG = '''run_name: {run_name}
seed: 42
dataset: {dataset}
task: classification_multiclass

model:
  name: patchtst_phm_base
  d_model: 128
  n_layers: 4
  n_heads: 4
  d_ff: 512
  dropout: 0.1
  patch_size: 16
  n_patches: 32

data:
  processed_root: /content/drive/MyDrive/fm_fl_phmd/processed
  dataset: {dataset}
  batch_size: 64
  batch_size_policy: adaptive_by_channels
  max_channel_batch: 512
  min_batch_size: 1
  num_workers: 2

training:
  max_epochs: 20
  lr_head: 0.001
  lr_backbone: {lr_backbone}
  weight_decay: 0.01
  amp: auto
  grad_clip_norm: 1.0
  log_every: 50
  eval_every_epochs: 1
  metric_for_best: macro_f1_val
  head_dropout: 0.1
  max_train_batches_per_epoch: null
  max_val_batches: null
  max_test_batches: null

paths:
  log_dir: /content/drive/MyDrive/fm_fl_phmd/logs/downstream/{dataset_lower}
  checkpoint_dir: /content/drive/MyDrive/fm_fl_phmd/checkpoints/downstream/{dataset_lower}
'''

LR_BACKBONE_POR_MODO = {
    "from_scratch":    0.0001,
    "linear_probing":  0.0001,
    "full_finetuning": 0.00001,
}


def make_config(dataset: str, mode: str):
    dataset_lower = dataset.lower()
    suffix = {
        "from_scratch":    "from_scratch",
        "linear_probing":  "linear_probing",
        "full_finetuning": "full_ft_lr1e-5",
    }[mode]
    run_name = f"downstream_{dataset_lower}_patchtst_base_{suffix}"
    cfg = BASE_CONFIG.format(
        run_name=run_name,
        dataset=dataset,
        dataset_lower=dataset_lower,
        lr_backbone=LR_BACKBONE_POR_MODO[mode],
    )
    cfg_path = Path(f"/tmp/cfg_{dataset_lower}_{mode}.yaml")
    cfg_path.write_text(cfg)
    return cfg_path, run_name


MODES = ["from_scratch", "linear_probing", "full_finetuning"]
print("Helpers cargados. LR backbone por modo:", LR_BACKBONE_POR_MODO)


In [ ]:
# Dry-run por cada dataset elegible: construye el classifier, lee primer
# batch sintetico y sale. Detecta problemas de config/carga rapidamente.
for dataset in ELIGIBLE:
    cfg_path, _ = make_config(dataset, "from_scratch")
    print(f"\n=== DRY-RUN {dataset} ===")
    r = subprocess.run(
        ["python", "-m", "training.train_downstream_classification",
         "--config", str(cfg_path), "--mode", "from_scratch", "--dry-run"],
        capture_output=True, text=True, timeout=600,
    )
    print(r.stdout[-3000:] if len(r.stdout) > 3000 else r.stdout)
    if r.returncode != 0:
        print(f"[WARN] dry-run de {dataset} retorno {r.returncode}")
        print(r.stderr[-2000:] if r.stderr else "")


In [ ]:
# Bucle principal: ELIGIBLE x 3 modos.
# - Reentrante: si <log_dir>/<mode>/run_info.json existe, SKIP.
# - Tolerante a fallos por dataset/modo: log y continua.
# - Stdout en vivo a /content y a Drive (_stdout/<mode>.stdout.log).
# - Renombra el output canonico (run_name) a <mode>/ al terminar OK.

results = []

for dataset in ELIGIBLE:
    log_root = LOG_DIR_ROOT / dataset.lower()
    ckpt_root = CKPT_DIR_ROOT / dataset.lower()
    log_root.mkdir(parents=True, exist_ok=True)
    ckpt_root.mkdir(parents=True, exist_ok=True)
    stdout_dir = log_root / "_stdout"
    stdout_dir.mkdir(exist_ok=True)

    for mode in MODES:
        cfg_path, run_name = make_config(dataset, mode)
        final_log = log_root / mode
        final_ckpt = ckpt_root / mode

        # REENTRANCIA
        if (final_log / "run_info.json").is_file():
            try:
                ri = json.load(open(final_log / "run_info.json"))
                tm = ri.get("test_metrics") or {}
                results.append({
                    "dataset": dataset, "mode": mode, "ok": True, "cached": True,
                    "macro_f1": tm.get("macro_f1"),
                    "bal_acc": tm.get("balanced_accuracy"),
                    "acc": tm.get("accuracy"),
                    "best_epoch": ri.get("best_epoch"),
                    "elapsed_seconds": ri.get("elapsed_seconds"),
                    "config_hash": ri.get("config_hash"),
                })
                print(f"  SKIP {dataset}/{mode}: ya hay run_info.json  "
                      f"(macro_f1={tm.get('macro_f1')})")
            except Exception as e:
                print(f"  WARN {dataset}/{mode}: run_info.json existe pero ilegible: {e}")
            continue

        canon_log = log_root / run_name
        canon_ckpt = ckpt_root / run_name
        ts = time.strftime("%Y%m%dT%H%M%S")
        for d in (canon_log, canon_ckpt):
            if d.exists():
                backup = d.with_name(d.name + f".stale-{ts}")
                print(f"  moviendo {d} previo a {backup}")
                shutil.move(str(d), str(backup))

        cmd = ["python", "-m", "training.train_downstream_classification",
               "--config", str(cfg_path), "--mode", mode]
        if mode != "from_scratch":
            cmd += ["--checkpoint", str(SSL_CKPT)]

        stdout_path = stdout_dir / f"{mode}.stdout.log"
        print(f"\n{'='*72}")
        print(f"[{time.strftime('%H:%M:%S')}] === START {dataset}/{mode} ===")
        print(f"  cfg:    {cfg_path}")
        print(f"  cmd:    {' '.join(cmd)}")
        print(f"  stdout: {stdout_path}")

        t0 = time.time()
        try:
            with open(stdout_path, "w") as f:
                proc = subprocess.Popen(
                    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                    text=True, bufsize=1,
                )
                for line in proc.stdout:
                    print(line, end="")
                    f.write(line); f.flush()
                proc.wait()
            rc = proc.returncode
        except Exception as e:
            rc = -999
            print(f"[EXCEPTION] {dataset}/{mode}: {e}")
        elapsed = time.time() - t0

        if rc != 0:
            print(f"[FAIL] {dataset}/{mode}: returncode={rc}  elapsed={elapsed:.1f}s")
            results.append({
                "dataset": dataset, "mode": mode, "ok": False,
                "returncode": rc, "elapsed_seconds": elapsed,
            })
            continue

        for src, dst in [(canon_log, final_log), (canon_ckpt, final_ckpt)]:
            if dst.exists():
                shutil.move(str(dst), str(dst.with_name(dst.name + f".prev-{ts}")))
            if src.exists():
                shutil.move(str(src), str(dst))
                print(f"  renombrado {src.name} -> {dst}")

        ri_path = final_log / "run_info.json"
        if not ri_path.is_file():
            print(f"[WARN] {dataset}/{mode}: terminado OK pero no hay run_info.json en {ri_path}")
            results.append({
                "dataset": dataset, "mode": mode, "ok": False,
                "error": "no_run_info", "elapsed_seconds": elapsed,
            })
            continue

        ri = json.load(open(ri_path))
        tm = ri.get("test_metrics") or {}
        results.append({
            "dataset": dataset, "mode": mode, "ok": True, "cached": False,
            "macro_f1": tm.get("macro_f1"),
            "bal_acc": tm.get("balanced_accuracy"),
            "acc": tm.get("accuracy"),
            "best_epoch": ri.get("best_epoch"),
            "elapsed_seconds": ri.get("elapsed_seconds"),
            "config_hash": ri.get("config_hash"),
        })
        print(f"[{time.strftime('%H:%M:%S')}] === DONE {dataset}/{mode}  "
              f"elapsed={elapsed:.1f}s  macro_f1={tm.get('macro_f1')} ===")

print(f"\n\nBucle terminado. {len(results)} entradas en results.")


In [ ]:
# Resumen final: tabla agrupada por dataset y agregada a Drive
print(f"\n\n{'='*78}\n[RESUMEN FINAL]\n{'='*78}")
print(f"{'dataset':<12} {'mode':<18} {'acc':>8} {'bal_acc':>10} {'macro_f1':>10}  {'best_ep':>7}  {'elapsed':>9}")
print("-" * 78)

def _by_mode(r):
    order = {"from_scratch": 0, "linear_probing": 1, "full_finetuning": 2}
    return (r["dataset"], order.get(r["mode"], 99))

for r in sorted(results, key=_by_mode):
    if not r.get("ok"):
        print(f"{r['dataset']:<12} {r['mode']:<18}  FAIL  returncode={r.get('returncode', r.get('error'))}  elapsed={r.get('elapsed_seconds', 0):.0f}s")
        continue
    acc = (r.get("acc") or 0)
    ba = (r.get("bal_acc") or 0)
    mf1 = (r.get("macro_f1") or 0)
    cached = " (cached)" if r.get("cached") else ""
    print(f"{r['dataset']:<12} {r['mode']:<18} {acc:>8.4f} {ba:>10.4f} {mf1:>10.4f}  "
          f"{str(r.get('best_epoch')):>7}  {str(round(r.get('elapsed_seconds', 0))):>8}s{cached}")

print(f"\n\n{'='*78}\n[MEJOR MODO POR DATASET]\n{'='*78}")
print(f"{'dataset':<12} {'mejor_modo':<18} {'macro_f1':>10}")
print("-" * 78)
by_ds = {}
for r in results:
    if not r.get("ok"):
        continue
    d = r["dataset"]
    if d not in by_ds or (r.get("macro_f1") or 0) > (by_ds[d].get("macro_f1") or 0):
        by_ds[d] = r
for d in sorted(by_ds):
    r = by_ds[d]
    print(f"{d:<12} {r['mode']:<18} {r.get('macro_f1', 0):>10.4f}")

summary_path = LOG_DIR_ROOT / "summary_5tt.json"
summary_path.write_text(
    json.dumps({"results": results, "ts": time.strftime("%Y-%m-%dT%H:%M:%S")},
               indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"\nResumen guardado en: {summary_path}")


In [ ]:
# Copiar run_infos al repo (ligeros, citables)
import shutil
from pathlib import Path

DST_ROOT = REPO / "results/downstream"
copiados = 0
for r in results:
    if not r.get("ok"):
        continue
    d = r["dataset"]
    mode = r["mode"]
    SRC = LOG_DIR_ROOT / d.lower() / mode
    DST = DST_ROOT / d.lower() / mode
    DST.mkdir(parents=True, exist_ok=True)
    for fn in ["run_info.json", "config.yaml", "label_mapping.json"]:
        src = SRC / fn
        if src.is_file():
            shutil.copy(src, DST / fn)
            copiados += 1

src_sum = LOG_DIR_ROOT / "summary_5tt.json"
if src_sum.is_file():
    DST_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copy(src_sum, DST_ROOT / "summary_5tt.json")
    copiados += 1

print(f"Archivos copiados al repo: {copiados}")
import subprocess
print(subprocess.run(["git", "status", "--short"], cwd=REPO, capture_output=True, text=True).stdout)


In [ ]:
# Commit + push final (controlado por AUTO_COMMIT / AUTO_PUSH).
#
# Por defecto AUTO_COMMIT=False y AUTO_PUSH=False -> solo se imprimen los
# comandos sugeridos. El usuario autoriza manualmente.
#
# Si quieres automatizar, cambia AUTO_COMMIT=True y AUTO_PUSH=True en la
# celda de setup (4) ANTES de ejecutar este notebook.
import subprocess
from pathlib import Path

ok_results = [r for r in results if r.get("ok")]
if not ok_results:
    print("No hay resultados ok que commitear. Salida limpia.")
else:
    lineas = [
        "results(downstream): baseline en TT primary (HSG18, PHM18, PBCP16)",
        "",
        "Tres modos (from_scratch, linear_probing, full_ft con lr_backbone=1e-5)",
        "sobre los TT primary classification del MVP. CALCE_CS2 e IEEE14 se",
        "filtraron por ser regresion (RUL). Misma config base que CWRU; full_ft",
        "con el sweet spot detectado en CWRU.",
        "",
        "Resumen test_macro_f1:",
    ]

    def _key(r):
        order = {"from_scratch": 0, "linear_probing": 1, "full_finetuning": 2}
        return (r["dataset"], order.get(r["mode"], 99))

    for r in sorted(results, key=_key):
        if r.get("ok"):
            lineas.append(f"  {r['dataset']:<12} {r['mode']:<18} "
                          f"macro_f1={r.get('macro_f1', 0):.4f}  "
                          f"bal_acc={r.get('bal_acc', 0):.4f}")
        else:
            lineas.append(f"  {r['dataset']:<12} {r['mode']:<18} FAIL")

    msg_path = Path("/tmp/commit_msg_5tt.txt")
    msg_path.write_text("\n".join(lineas) + "\n", encoding="utf-8")

    if AUTO_COMMIT:
        def sh(cmd):
            r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
            print("$", " ".join(cmd))
            if r.stdout: print(r.stdout)
            if r.stderr: print(r.stderr)
            return r.returncode

        sh(["git", "add", "results/downstream/"])
        sh(["git", "commit", "-F", str(msg_path)])
        if AUTO_PUSH:
            sh(["git", "push", "origin", "feature/exploration_phmd"])
        else:
            print("\nAUTO_PUSH=False: NO se ha pusheado. Comando sugerido:")
            print("  git push origin feature/exploration_phmd")
        sh(["git", "log", "--oneline", "-5"])
    else:
        print("AUTO_COMMIT=False: NO se ha commiteado.")
        print(f"Mensaje sugerido escrito en: {msg_path}")
        print("Comandos sugeridos para autorizar manualmente:")
        print("  cd /content/fm_fl_phmd")
        print("  git add results/downstream/")
        print(f"  git commit -F {msg_path}")
        if AUTO_PUSH:
            print("  (AUTO_PUSH=True pero AUTO_COMMIT=False; push se ignora)")
        print("  git push origin feature/exploration_phmd  # solo si autorizas")
